# Notebook 1: exploración, refinado y modelo de datos

**Obligatorio Herramientas de Software para Big Data, Parte 1. Vuelos regulares de Brasil, 2025.**
Grupo: Pilar Fernández, Jerónimo Brum y Fernando Tonarelli.

Este notebook va desde los archivos crudos que dejó NiFi en la zona *landing* hasta las tablas del modelo creadas y pobladas en Hive. El notebook 2 (analista) arranca de esas tablas y no lee nada de HDFS directamente.

| Paso | Qué hace | Salida |
|---|---|---|
| 1 | Lectura de las 6 fuentes desde `/lnd` | DataFrames crudos |
| 2 | Exploración: filas, columnas, esquema, nulos, duplicados, claves | Perfil de cada tabla |
| 3 | Validaciones a mano | Conteos contra los valores conocidos |
| 4 | Reglas de negocio y refinado | Tablas limpias |
| 5 | Guardado en `/rfn` y control de filas | Parquet en la zona refinada |
| 6 | Modelo estrella | Hechos y dimensiones |
| 7 | DDL de Hive y carga | Base `vuelos_brasil` |
| 8 | Validación final en Hive | Conteos, particiones, tipo de tabla |

Todo se hace con Spark. Pandas no se usa en este notebook.

## 0. Configuración

Antes de correr, en la terminal de la VM:

```
docker exec namenode hdfs dfs -chmod -R 777 /rfn
```

Hive y Spark escriben con usuarios distintos; sin ese permiso, la escritura en `/rfn` falla.

**Lo único que hay que cambiar al pasar el notebook a otra VM es `INGESTA`**: la fecha de la carpeta `ingesta=AAAAMMDD` que creó NiFi (se ve con `hdfs dfs -ls /lnd/vuelos_brasil/anac_vra`).

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("nb1_vuelos_brasil_refinado")
         .enableHiveSupport()
         .getOrCreate())

INGESTA = "20261111"      # carpeta ingesta=AAAAMMDD que dejó NiFi en landing
VERSION = "v1"            # versión del refinado; cambia si cambian las reglas
PREFIX = ""               # vacío = HDFS por defecto de la VM
ENCODING_AERODROMOS = "ISO-8859-1"   # el cadastro de aeródromos viene en Latin-1

LND = f"{PREFIX}/lnd/vuelos_brasil"
RFN = f"{PREFIX}/rfn/vuelos_brasil/{VERSION}"
MODELO = f"{RFN}/modelo"
DB = "vuelos_brasil"

def ruta_lnd(fuente, archivo="*"):
    return f"{LND}/{fuente}/ingesta={INGESTA}/{archivo}"

print("Landing :", LND)
print("Refined :", RFN)
print("Modelo  :", MODELO)

## 1. Lectura de las fuentes

| Fuente | Carpeta en landing | Formato |
|---|---|---|
| Vuelos (ANAC VRA), un CSV por mes | `anac_vra` | `;`, UTF-8 con BOM, línea "Atualizado em" antes del encabezado |
| Empresas aéreas brasileñas (ANAC) | `anac_empresas_br` | igual que los vuelos |
| Empresas aéreas extranjeras (ANAC) | `anac_empresas_ext` | igual que los vuelos |
| Aeródromos públicos (ANAC) | `anac_aerodromos` | `;` sin comillas, **Latin-1**, línea "Atualizado em" |
| Aeropuertos del mundo (OurAirports) | `ourairports_airports` | `,` con comillas, UTF-8, encabezado en la línea 1 |
| Países (OurAirports) | `ourairports_countries` | igual que aeropuertos |

Los archivos de ANAC tienen una primera línea `Atualizado em: <fecha>` y recién la segunda es el encabezado. Spark no tiene una opción para saltear líneas, así que `leer_anac` lee todo sin encabezado, toma el encabezado de la primera fila que no es "Atualizado em" y descarta esa fila y la del encabezado de cada archivo. De paso guarda la fecha de actualización de cada archivo.

Las columnas se renombran a `snake_case` (Hive pasa todo a minúscula y no acepta espacios ni tildes).

In [ ]:
def leer_anac(ruta, columnas, encoding="UTF-8", quote='"'):
    # Lee un CSV de ANAC: descarta la línea "Atualizado em" y el encabezado de cada archivo.
    # La primera línea ("Atualizado em") tiene una sola columna; si se dejara inferir el esquema,
    # Spark leería todo el archivo con una columna. Por eso se pasa el esquema explícito.
    esquema = T.StructType([T.StructField(f"_c{i}", T.StringType()) for i in range(len(columnas))])
    crudo = (spark.read
             .schema(esquema)
             .option("sep", ";")
             .option("header", False)
             .option("quote", quote)
             .option("encoding", encoding)
             .csv(ruta)
             .withColumn("archivo_origen", F.regexp_extract(F.input_file_name(), r"([^/]+)$", 1)))
    # Spark lee los campos vacíos como nulos: las comparaciones tienen que ser null-safe,
    # si no se pierden las filas con la primera columna vacía.
    es_actualizado = F.coalesce(F.col("_c0").contains("Atualizado em"), F.lit(False))
    encabezado = crudo.filter(~es_actualizado).first()
    n_cols = len(columnas)
    assert encabezado[n_cols - 1] is not None, "El encabezado tiene menos columnas que las esperadas"
    print("Encabezado original:", [encabezado[i] for i in range(n_cols)])
    print("Actualización:", [r[0].replace("﻿", "").strip()
                             for r in crudo.filter(es_actualizado).select("_c0").distinct().collect()])
    datos = crudo.filter(~es_actualizado & ~F.col("_c0").eqNullSafe(encabezado[0]))
    for i, nombre in enumerate(columnas):
        datos = datos.withColumnRenamed(f"_c{i}", nombre)
    return datos

COLS_VRA = ["icao_empresa", "numero_voo", "codigo_di", "tipo_linha", "icao_origem", "icao_destino",
            "partida_prevista", "partida_real", "chegada_prevista", "chegada_real",
            "situacao_voo", "codigo_justificativa"]
COLS_EMPRESAS = ["icao", "iata", "razao_social", "servico", "endereco", "cidade", "uf", "cep",
                 "telefone", "email", "instrumento", "data_instrumento", "ativa"]
COLS_AERODROMOS = ["codigo_oaci", "ciad", "nome", "municipio", "uf", "municipio_servido", "uf_servido",
                   "lat_geopoint", "lon_geopoint", "latitude_gms", "longitude_gms", "altitude",
                   "operacao_diurna", "operacao_noturna", "situacao", "validade_registro",
                   "portaria_registro", "link_portaria"]

In [ ]:
vuelos_raw = leer_anac(ruta_lnd("anac_vra", "*.csv"), COLS_VRA)
emp_br_raw = leer_anac(ruta_lnd("anac_empresas_br", "*.csv"), COLS_EMPRESAS)
emp_ext_raw = leer_anac(ruta_lnd("anac_empresas_ext", "*.csv"), COLS_EMPRESAS)
aerodromos_raw = leer_anac(ruta_lnd("anac_aerodromos", "*.csv"), COLS_AERODROMOS,
                           encoding=ENCODING_AERODROMOS, quote="")

def leer_ourairports(ruta):
    return (spark.read
            .option("header", True)
            .option("quote", '"')
            .option("escape", '"')
            .option("multiLine", True)
            .csv(ruta))

airports_raw = leer_ourairports(ruta_lnd("ourairports_airports", "airports.csv"))
countries_raw = leer_ourairports(ruta_lnd("ourairports_countries", "countries.csv"))

## 2. Exploración de cada tabla

Para cada tabla: primeras filas, cantidad de filas y columnas, esquema, nulos o vacíos por columna, filas duplicadas y unicidad de la clave. En esta etapa todas las columnas son texto: los tipos se asignan en el refinado.

`valores_basura` cuenta valores que en estas fuentes significan "sin dato" aunque no estén vacíos ("N/I", "..", "000", "null", etc.).

In [ ]:
BASURA = ["N/I", "N/A", "..", ".", "-", "null", "0", "00", "000", "0000000", "00.000-000", "***", "**"]

def perfilar(df, nombre, clave=None, basura=True):
    n = df.count()
    print(f"==================== {nombre} ====================")
    print(f"Filas: {n:,} | Columnas: {len(df.columns)}")
    df.printSchema()
    df.show(5, truncate=40)
    nulos = df.select([F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1)
                             .otherwise(0)).alias(c) for c in df.columns]).first().asDict()
    filas = []
    for c in df.columns:
        b = (df.filter(F.trim(F.col(c).cast("string")).isin(BASURA)).count() if basura else 0)
        filas.append((c, int(nulos[c] or 0), round(100 * (nulos[c] or 0) / max(n, 1), 2), b))
    spark.createDataFrame(filas, "columna string, nulos_o_vacios long, pct double, valores_basura long") \
         .show(len(filas), truncate=False)
    print("Filas duplicadas completas:", n - df.dropDuplicates().count())
    if clave:
        rep = df.filter(F.concat_ws("", *clave) != "").groupBy(*clave).count().filter("count > 1")
        print(f"Valores de la clave {clave} repetidos:", rep.count())
        rep.orderBy(F.desc("count")).show(5, truncate=False)

### 2.1 Vuelos (VRA)

| Columna | Significado |
|---|---|
| icao_empresa | Código ICAO de la aerolínea (3 letras) |
| numero_voo | Número de vuelo, con ceros adelante ("0057") |
| codigo_di | Código de autorización (DI); 0 en la gran mayoría |
| tipo_linha | N nacional, I internacional, C carga doméstica, G carga internacional |
| icao_origem, icao_destino | Código ICAO de los aeropuertos de salida y llegada |
| partida_prevista, partida_real | Salida programada y real, en hora de Brasília |
| chegada_prevista, chegada_real | Llegada programada y real, en hora de Brasília |
| situacao_voo | REALIZADO o CANCELADO |
| codigo_justificativa | Motivo de atraso o cancelación; "N/A" si no hay |

Clave candidata del vuelo: empresa + número de vuelo + fecha y hora de referencia + origen + destino. La fecha de referencia es la salida prevista y, si no existe, la salida real (ver la regla 2 de la sección 4).

In [ ]:
perfilar(vuelos_raw, "VUELOS (VRA)", basura=False)
vuelos_raw.groupBy("archivo_origen").count().orderBy("archivo_origen").show(12)
vuelos_raw.groupBy("situacao_voo").count().orderBy(F.desc("count")).show()
vuelos_raw.groupBy("tipo_linha").count().orderBy(F.desc("count")).show()
vuelos_raw.groupBy("codigo_justificativa").count().orderBy(F.desc("count")).show(10)
vuelos_raw.groupBy("codigo_di").count().orderBy(F.desc("count")).show(15)
# Formato de las fechas: algunas traen fracción de segundo
cols_fecha = ["partida_prevista", "partida_real", "chegada_prevista", "chegada_real"]
for c in cols_fecha:
    print(f"{c}: con fracción de segundo = {vuelos_raw.filter(F.col(c).contains('.')).count():,}"
          f" | con el texto 'null' = {vuelos_raw.filter(F.col(c) == 'null').count():,}")
print("Vuelos sin salida prevista ('null'), por situación, tipo y código DI:")
sin_prevista = vuelos_raw.filter(F.col("partida_prevista") == "null")
sin_prevista.groupBy("situacao_voo").count().show()
sin_prevista.groupBy("tipo_linha").count().orderBy(F.desc("count")).show()
sin_prevista.groupBy("codigo_di").count().orderBy(F.desc("count")).show()
print("Cancelados con salida real informada:",
      vuelos_raw.filter((F.col("situacao_voo") == "CANCELADO") & (F.col("partida_real") != "null")).count())

### 2.2 Empresas aéreas brasileñas y extranjeras

Los dos archivos tienen las mismas 13 columnas.

| Columna | Significado |
|---|---|
| icao | Código ICAO de 3 letras; solo lo tienen las empresas de transporte aéreo |
| iata | Pese a que el archivo la llama "Estrangeira", es el código IATA (G3 Gol, AD Azul) |
| razao_social | Razón social |
| servico | Servicios autorizados (en extranjeras: REGULAR, NÃO REGULAR u OFFLINE) |
| endereco, cidade, uf, cep | Dirección de la sede o del representante en Brasil |
| telefone, email | Contacto (dato personal en algunos casos: no pasa a refined) |
| instrumento, data_instrumento | Portaria que habilita a la empresa y su fecha (M/d/yyyy h:mm:ss AM) |
| ativa | Estado; siempre ATIVA |

In [ ]:
perfilar(emp_br_raw, "EMPRESAS BRASILEÑAS", clave=["icao"])
perfilar(emp_ext_raw, "EMPRESAS EXTRANJERAS", clave=["icao"])
print("ICAO en los dos archivos a la vez:",
      emp_br_raw.select("icao").filter("icao != ''").intersect(emp_ext_raw.select("icao")).count())

### 2.3 Aeródromos públicos (ANAC)

| Columna | Significado |
|---|---|
| codigo_oaci | Código ICAO de 4 letras (OACI en portugués) |
| ciad | Identificador del aeródromo en Brasil: sigla del estado + 4 dígitos |
| nome, municipio, uf | Nombre, municipio y estado (nombre completo, no sigla) |
| municipio_servido, uf_servido | Ciudad y estado a los que da servicio |
| lat_geopoint, lon_geopoint | Coordenadas en grados decimales |
| latitude_gms, longitude_gms | Las mismas coordenadas en grados, minutos y segundos |
| altitude | Altura en metros, con coma decimal |
| operacao_diurna, operacao_noturna | Tipo de operación permitida |
| situacao | Cadastrado o Interditado |
| validade_registro, portaria_registro, link_portaria | Datos del registro |

In [ ]:
perfilar(aerodromos_raw, "AERÓDROMOS PÚBLICOS", clave=["codigo_oaci"], basura=False)
aerodromos_raw.groupBy("uf").count().orderBy(F.desc("count")).show(30, truncate=False)
aerodromos_raw.groupBy("situacao").count().show()

### 2.4 Aeropuertos y países (OurAirports)

Diccionario oficial: https://ourairports.com/help/data-dictionary.html. Las columnas que se usan son `ident`, `gps_code`, `icao_code` (códigos), `type` (tamaño), `name`, `iso_country`, `iso_region` (estado, por ejemplo BR-SP), `municipality` y las coordenadas. En `countries`: `code` (ISO de 2 letras), `name` y `continent`.

Ojo: en `continent`, "NA" es América del Norte, y en `countries.code`, "NA" es Namibia. Spark lo lee como texto; pandas lo convertiría en nulo.

In [ ]:
perfilar(airports_raw, "AEROPUERTOS (OurAirports)", clave=["ident"], basura=False)
airports_raw.filter("iso_country = 'BR'").groupBy("type").count().orderBy(F.desc("count")).show()
for c in ["ident", "gps_code", "icao_code"]:
    rep = airports_raw.filter(F.col(c).isNotNull()).groupBy(c).count().filter("count > 1").count()
    print(f"{c}: valores repetidos = {rep}")
perfilar(countries_raw, "PAÍSES (OurAirports)", clave=["code"], basura=False)

## 3. Validaciones a mano

Se comparan los conteos con valores revisados a mano en los archivos originales (abiertos en el navegador o en un editor) el 10 y 11/11. Si un valor no coincide, se marca **DIFERENTE** y hay que revisar antes de seguir.

Además se prueba cómo cruzan los vuelos con las otras tablas: esto decide qué columna de OurAirports usar como clave.

In [ ]:
ESPERADOS_VRA = {"VRA_20251.csv": 89616}           # agregar los demás meses al revisarlos
ESPERADO_SITUACAO_ENERO = {"REALIZADO": 86376, "CANCELADO": 3240}
ESPERADO_TIPO_ENERO = {"N": 70984, "I": 15266, "G": 1808, "C": 1558}

def chequear(nombre, obtenido, esperado):
    estado = "OK" if obtenido == esperado else "DIFERENTE"
    print(f"[{estado}] {nombre}: obtenido {obtenido:,} / esperado {esperado:,}")

por_archivo = {r["archivo_origen"]: r["count"] for r in vuelos_raw.groupBy("archivo_origen").count().collect()}
for archivo, esperado in ESPERADOS_VRA.items():
    if archivo in por_archivo:
        chequear(f"vuelos en {archivo}", por_archivo[archivo], esperado)

enero = vuelos_raw.filter(F.col("archivo_origen") == "VRA_20251.csv")
if enero.limit(1).count():
    for k, v in ESPERADO_SITUACAO_ENERO.items():
        chequear(f"enero, situacao {k}", enero.filter(F.col("situacao_voo") == k).count(), v)
    for k, v in ESPERADO_TIPO_ENERO.items():
        chequear(f"enero, tipo_linha {k}", enero.filter(F.col("tipo_linha") == k).count(), v)

chequear("empresas brasileñas", emp_br_raw.count(), 729)
chequear("aeródromos públicos", aerodromos_raw.count(), 496)
print("Empresas extranjeras:", emp_ext_raw.count(), "(se revisaron unas 149 a mano)")
print("Aeropuertos OurAirports:", airports_raw.count(), "(86.351 el 11/11; cambia todos los días)")
print("Países OurAirports:", countries_raw.count(), "(249 el 11/11)")

Un vuelo concreto revisado a mano en el CSV original: el primer registro de enero es Air Europa (AEA) 0057, de Madrid (LEMD) a Guarulhos (SBGR), previsto 2025-01-29 19:55, salió 20:45. El atraso calculado a mano es 50 minutos. La misma cuenta se repite con Spark en el refinado (sección 4).

In [ ]:
vuelos_raw.filter((F.col("icao_empresa") == "AEA") & (F.col("numero_voo") == "0057")
                  & (F.col("partida_prevista") == "2025-01-29 19:55:00")).show(truncate=False)

In [ ]:
# Cruce de los vuelos con las otras tablas
empresas_icao = emp_br_raw.select("icao").union(emp_ext_raw.select("icao")).filter("icao != ''").distinct()
emp_vuelos = vuelos_raw.groupBy("icao_empresa").count()
sin_empresa = emp_vuelos.join(empresas_icao, emp_vuelos.icao_empresa == empresas_icao.icao, "left_anti")
print("Empresas en los vuelos:", emp_vuelos.count(), "| sin datos en ANAC:", sin_empresa.count())
sin_empresa.orderBy(F.desc("count")).show(truncate=False)

aerop_vuelos = (vuelos_raw.select(F.col("icao_origem").alias("icao"))
                .union(vuelos_raw.select(F.col("icao_destino").alias("icao")))
                .groupBy("icao").count())
print("Aeropuertos distintos en los vuelos:", aerop_vuelos.count())
for c in ["gps_code", "ident", "icao_code"]:
    claves = airports_raw.select(F.col(c).alias("icao")).filter(F.col("icao").isNotNull())
    faltan = aerop_vuelos.join(claves, "icao", "left_anti")
    print(f"Cruce por {c}: no cruzan {faltan.count()} aeropuertos, "
          f"{faltan.agg(F.sum('count')).first()[0] or 0} movimientos")
anac = aerodromos_raw.select(F.col("codigo_oaci").alias("icao"))
print("Aeropuertos de los vuelos que están en el cadastro de ANAC:", aerop_vuelos.join(anac, "icao").count())

### Hallazgos de la exploración (enero 2025; revisar con el año completo)

| Hallazgo | Consecuencia |
|---|---|
| Las fechas sin dato vienen con el **texto "null"**, no vacías | Se tratan como nulo en el refinado |
| 2.253 vuelos realizados **no tienen salida prevista**; todos tienen código DI distinto de 0 (vuelos no programados) | No tienen atraso: quedan fuera de la puntualidad, pero cuentan como realizados. La fecha del vuelo es la salida real |
| Unas 2.700 salidas y llegadas previstas tienen **fracción de segundo** ("01:40:00.100000000") | Se toma la fecha hasta los segundos |
| La clave empresa + número + salida prevista + origen **se repetía 825 veces**, por los vuelos sin hora prevista | La clave usa la fecha de referencia y agrega el destino: así no se repite |
| **Código Justificativa** es "N/A" en todos los vuelos | La columna no sirve para explicar cancelaciones |
| El archivo de enero trae unos pocos vuelos con fecha de febrero | Quedan en la partición de su propio mes; al cargar los 12 meses se verifica que no se dupliquen |
| Las listas de empresas de ANAC tienen **solo las activas hoy**: PTB (VoePass) y PAM no están | Sus vuelos se mantienen con nacionalidad DESCONOCIDA |
| Solo 20 de 729 empresas brasileñas tienen código ICAO | Las demás (taxis aéreos, fumigación) no aparecen en los vuelos |
| Los aeropuertos cruzan mejor con **gps_code** de OurAirports que con ident o icao_code | Se usa gps_code y, si no cruza, ident |
| Teléfono y email de las empresas incluyen mails personales | No pasan a la zona refinada |

## 4. Reglas de negocio y refinado

| # | Regla | Dónde se aplica |
|---|---|---|
| 1 | Clave del vuelo: empresa + número + fecha de referencia + origen + destino. Se verifica que no se repita; solo se eliminan filas duplicadas completas | vuelos |
| 2 | Las fechas vienen con el texto "null" cuando no hay dato. Los cancelados no tienen hora real: su atraso queda nulo y no se imputa. Los vuelos no programados (código DI distinto de 0) no tienen hora prevista: no tienen atraso y quedan fuera de la puntualidad, pero cuentan como realizados. La fecha de referencia del vuelo es la prevista o, si falta, la real. "No informado" queda fuera de las tasas | vuelos |
| 3 | Atraso = hora real − hora prevista, en minutos. Negativos (adelantos) se mantienen. Más de 24 h en valor absoluto se marca `atraso_extremo` y el atraso pasa a nulo | vuelos |
| 4 | Solo vuelos de pasajeros N e I en el modelo. En refined quedan todos, con el tipo | vuelos → fact |
| 5 | Empresas brasileñas y extranjeras en una sola tabla con `nacionalidad`. Una empresa de los vuelos que no está en ANAC queda como DESCONOCIDA, sin borrar sus vuelos | empresas → dim |
| 6 | Aeropuertos: se cruzan con OurAirports por `gps_code` (y si no, por `ident`). Lo que no cruza queda con tamaño DESCONOCIDO | dim_aeropuerto |
| 7 | Limpieza: trim, valores basura a nulo, tipos correctos, Latin-1 en aeródromos. Teléfono, email y dirección no pasan a refined (datos personales) | todas |
| 8 | Hora local: ANAC informa en hora de Brasília. Brasil no tiene horario de verano desde 2019, así que cada aeropuerto brasileño tiene un desfase fijo según su estado | dim_aeropuerto, fact |

In [ ]:
def limpiar(df, columnas, nulos=("",)):
    for c in columnas:
        t = F.trim(F.col(c))
        df = df.withColumn(c, F.when(t.isNull() | t.isin(list(nulos)), None).otherwise(t))
    return df

# Algunas fechas vienen con fracción de segundo ("2025-01-04 01:40:00.100000000"); se toman los
# primeros 19 caracteres (hasta los segundos), que es la precisión del resto del archivo.
ts = lambda c: F.to_timestamp(F.substring(F.col(c), 1, 19), "yyyy-MM-dd HH:mm:ss")
minutos = lambda a, b: (F.unix_timestamp(a) - F.unix_timestamp(b)) / 60.0

vuelos = (limpiar(vuelos_raw, COLS_VRA, nulos=("", "N/A", "null"))
          .withColumn("situacao_voo", F.upper("situacao_voo"))
          .withColumn("partida_prevista", ts("partida_prevista"))
          .withColumn("partida_real", ts("partida_real"))
          .withColumn("chegada_prevista", ts("chegada_prevista"))
          .withColumn("chegada_real", ts("chegada_real"))
          .withColumn("fecha_referencia", F.coalesce("partida_prevista", "partida_real"))
          .withColumn("tiene_horario_previsto", F.col("partida_prevista").isNotNull())
          .withColumn("realizado", F.col("situacao_voo") == "REALIZADO")
          .withColumn("cancelado", F.col("situacao_voo") == "CANCELADO")
          .withColumn("informado", F.col("realizado") | F.col("cancelado"))
          .withColumn("atraso_partida_min", F.when(F.col("realizado"), minutos("partida_real", "partida_prevista")))
          .withColumn("atraso_chegada_min", F.when(F.col("realizado"), minutos("chegada_real", "chegada_prevista")))
          .withColumn("atraso_extremo", (F.abs("atraso_partida_min") > 1440) | (F.abs("atraso_chegada_min") > 1440))
          .withColumn("atraso_extremo", F.coalesce("atraso_extremo", F.lit(False)))
          .withColumn("atraso_partida_min", F.when(~F.col("atraso_extremo"), F.col("atraso_partida_min")))
          .withColumn("atraso_chegada_min", F.when(~F.col("atraso_extremo"), F.col("atraso_chegada_min")))
          .withColumn("anio_mes", F.date_format("fecha_referencia", "yyyy-MM")))

clave_vuelo = ["icao_empresa", "numero_voo", "fecha_referencia", "icao_origem", "icao_destino"]
antes = vuelos.count()
vuelos = vuelos.dropDuplicates([c for c in vuelos.columns if c != "archivo_origen"])
print(f"Filas: {antes:,} | duplicados completos eliminados: {antes - vuelos.count():,}")
repetidas = vuelos.groupBy(*clave_vuelo).count().filter("count > 1")
print("Claves de vuelo repetidas:", repetidas.count())
repetidas.show(5, truncate=False)
print("Vuelos sin fecha de referencia:", vuelos.filter(F.col("fecha_referencia").isNull()).count())
print("Vuelos sin horario previsto (no programados):", vuelos.filter(~F.col("tiene_horario_previsto")).count())
print("Vuelos con atraso extremo (> 24 h):", vuelos.filter("atraso_extremo").count())
vuelos.groupBy("situacao_voo", "informado").count().show()
vuelos.select(F.min("atraso_partida_min"), F.expr("percentile_approx(atraso_partida_min, 0.5)"),
              F.avg("atraso_partida_min"), F.max("atraso_partida_min")).show()

Validación a mano del atraso: el vuelo AEA 0057 tiene que dar 50 minutos de atraso de salida.

In [ ]:
vuelos.filter((F.col("icao_empresa") == "AEA") & (F.col("numero_voo") == "0057")
              & (F.col("partida_prevista") == F.lit("2025-01-29 19:55:00").cast("timestamp"))) \
      .select("icao_empresa", "numero_voo", "partida_prevista", "partida_real", "atraso_partida_min").show()

In [ ]:
# Empresas: una sola tabla, sin datos personales
NULOS_EMP = ("", "N/I", "..", ".", "-", "null", "***", "**", "0", "00", "000", "0000000", "00.000-000")
def refinar_empresas(df, nacionalidad):
    return (limpiar(df, COLS_EMPRESAS, NULOS_EMP)
            .withColumn("nacionalidad", F.lit(nacionalidad))
            .withColumn("data_instrumento",
                        F.to_date(F.to_timestamp("data_instrumento", "M/d/yyyy h:mm:ss a")))
            .select(F.col("icao").alias("icao_empresa"), F.col("iata").alias("iata_empresa"),
                    "razao_social", "servico", "nacionalidad", "instrumento", "data_instrumento", "ativa"))

empresas = refinar_empresas(emp_br_raw, "BRASILEÑA").unionByName(refinar_empresas(emp_ext_raw, "EXTRANJERA"))
empresas.groupBy("nacionalidad", F.col("icao_empresa").isNotNull().alias("tiene_icao")).count().show()

In [ ]:
# Aeródromos de ANAC: estado como sigla, coordenadas y altitud numéricas
UF_SIGLA = {"ACRE": "AC", "ALAGOAS": "AL", "AMAPA": "AP", "AMAZONAS": "AM", "BAHIA": "BA", "CEARA": "CE",
            "DISTRITO FEDERAL": "DF", "ESPIRITO SANTO": "ES", "GOIAS": "GO", "MARANHAO": "MA",
            "MATO GROSSO": "MT", "MATO GROSSO DO SUL": "MS", "MINAS GERAIS": "MG", "PARA": "PA",
            "PARAIBA": "PB", "PARANA": "PR", "PERNAMBUCO": "PE", "PIAUI": "PI", "RIO DE JANEIRO": "RJ",
            "RIO GRANDE DO NORTE": "RN", "RIO GRANDE DO SUL": "RS", "RONDONIA": "RO", "RORAIMA": "RR",
            "SANTA CATARINA": "SC", "SAO PAULO": "SP", "SERGIPE": "SE", "TOCANTINS": "TO"}
mapa_uf = F.create_map(*[F.lit(x) for kv in UF_SIGLA.items() for x in kv])
sin_tildes = lambda c: F.upper(F.translate(F.col(c), "ÁÀÂÃÉÊÍÓÔÕÚÇáàâãéêíóôõúç", "AAAAEEIOOOUCaaaaeeiooouc"))

aerodromos = (limpiar(aerodromos_raw, COLS_AERODROMOS)
              .withColumn("uf_sigla", F.coalesce(mapa_uf[sin_tildes("uf")], mapa_uf[sin_tildes("uf_servido")]))
              .withColumn("latitud", F.col("lat_geopoint").cast("double"))
              .withColumn("longitud", F.col("lon_geopoint").cast("double"))
              .withColumn("altitud_m", F.regexp_replace("altitude", ",", ".").cast("double"))
              .select("codigo_oaci", "ciad", "nome", "municipio", "uf", "uf_sigla", "municipio_servido",
                      "uf_servido", "latitud", "longitud", "altitud_m", "operacao_diurna",
                      "operacao_noturna", "situacao"))
print("Aeródromos sin sigla de estado:", aerodromos.filter("uf_sigla IS NULL").count())
aerodromos.filter("uf_sigla IS NULL").select("codigo_oaci", "nome", "uf", "uf_servido").show(truncate=False)

In [ ]:
# OurAirports: solo las columnas útiles, con tipos
airports = (airports_raw
            .select(F.col("id").cast("long").alias("id_ourairports"), "ident", "type", "name",
                    F.col("latitude_deg").cast("double").alias("latitud"),
                    F.col("longitude_deg").cast("double").alias("longitud"),
                    F.col("elevation_ft").cast("int").alias("elevacion_ft"),
                    "continent", "iso_country", "iso_region", "municipality", "scheduled_service",
                    "gps_code", "icao_code", "iata_code"))
countries = countries_raw.select(F.col("code").alias("iso_pais"), F.col("name").alias("nombre_pais"), "continent")

## 5. Guardado en la zona refinada y control de filas

Todo en parquet bajo `/rfn/vuelos_brasil/v1/`. Después se vuelve a leer cada tabla y se compara la cantidad de filas con la del DataFrame, como en el Práctico 4.

In [ ]:
refinadas = {"vuelos": vuelos, "empresas": empresas, "aerodromos": aerodromos,
             "aeropuertos_mundo": airports, "paises": countries}

for nombre, df in refinadas.items():
    df.write.mode("overwrite").parquet(f"{RFN}/{nombre}")

for nombre, df in refinadas.items():
    escrito, leido = df.count(), spark.read.parquet(f"{RFN}/{nombre}").count()
    print(f"[{'OK' if escrito == leido else 'DIFERENTE'}] {nombre}: {escrito:,} filas escritas, {leido:,} leídas")

## 6. Modelo estrella

Las cinco preguntas son agregaciones de vuelos cortadas por empresa, aeropuerto, país y tiempo, así que se eligió un **modelo estrella**: una tabla de hechos con un vuelo por fila y cuatro dimensiones. Resuelve cada pregunta con pocos joins. El modelo normalizado obligaría a más joins para lo mismo; Data Vault agrega complejidad que no hace falta para una carga anual; y OBT repetiría los datos de empresa y aeropuerto en cada vuelo.

| Tabla | Grano | Contenido |
|---|---|---|
| `fact_vuelo` | un vuelo programado de pasajeros (N o I) | claves, horarios, atrasos, indicadores; particionada por `anio_mes` |
| `dim_empresa` | una aerolínea | ICAO, IATA, nombre, nacionalidad |
| `dim_aeropuerto` | un aeropuerto (se usa como origen y como destino) | nombre, tamaño, ciudad, estado, país, huso horario |
| `dim_pais` | un país | ISO, nombre, continente |
| `dim_fecha` | un día | año, mes, trimestre, día de la semana, semana |

Qué usa cada pregunta:

| Pregunta | Tablas | Columnas clave |
|---|---|---|
| P1 puntualidad por aerolínea, T1 contra T4, nacional e internacional | fact, dim_empresa, dim_fecha | puntual, realizado, tipo_linha, trimestre |
| P2 rutas con más cancelaciones y su concentración | fact, dim_empresa | icao_origem, icao_destino, cancelado, informado |
| P3 atraso por hora y día en los 10 aeropuertos con más movimiento | fact, dim_aeropuerto | hora_local, dia_semana_local, atraso_partida_min |
| P4 frecuencias por país, T1 contra T4, brasileñas contra extranjeras | fact, dim_aeropuerto, dim_pais, dim_empresa, dim_fecha | iso_pais, nacionalidad, trimestre, realizado |
| P5 puntualidad y cancelación por tamaño de aeropuerto y mes | fact, dim_aeropuerto | tamano, anio_mes, puntual, cancelado |

In [ ]:
# dim_empresa: empresas de ANAC con ICAO + las de los vuelos que no están (DESCONOCIDA)
w = Window.partitionBy("icao_empresa").orderBy(F.when(F.col("nacionalidad") == "BRASILEÑA", 0).otherwise(1))
dim_empresa = (empresas.filter(F.col("icao_empresa").isNotNull())
               .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
               .select("icao_empresa", "iata_empresa", F.col("razao_social").alias("nombre_empresa"),
                       "nacionalidad", "servico"))
faltantes = (vuelos.select("icao_empresa").distinct()
             .join(dim_empresa.select("icao_empresa"), "icao_empresa", "left_anti")
             .select("icao_empresa", F.lit(None).cast("string").alias("iata_empresa"),
                     F.lit(None).cast("string").alias("nombre_empresa"),
                     F.lit("DESCONOCIDA").alias("nacionalidad"), F.lit(None).cast("string").alias("servico")))
dim_empresa = dim_empresa.unionByName(faltantes)
dim_empresa.groupBy("nacionalidad").count().show()
faltantes.show()

In [ ]:
# dim_aeropuerto: códigos de los vuelos + OurAirports (gps_code, si no ident) + cadastro de ANAC
codigos = (vuelos.select(F.col("icao_origem").alias("icao_aeropuerto"))
           .union(vuelos.select(F.col("icao_destino").alias("icao_aeropuerto"))).distinct())

claves_oa = (airports.filter(F.col("gps_code").isNotNull()).withColumn("clave", F.col("gps_code")).withColumn("prio", F.lit(1))
             .unionByName(airports.withColumn("clave", F.col("ident")).withColumn("prio", F.lit(2))))
w = Window.partitionBy("clave").orderBy("prio")
claves_oa = claves_oa.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn", "prio")

TAMANO = {"large_airport": "GRANDE", "medium_airport": "MEDIANO", "small_airport": "CHICO"}
mapa_tamano = F.create_map(*[F.lit(x) for kv in TAMANO.items() for x in kv])
# Husos fijos de 2025. Por defecto UTC-3 (hora de Brasília).
HUSO_UF = {"AM": -4, "MT": -4, "MS": -4, "RO": -4, "RR": -4, "AC": -5}
HUSO_EXCEPCION = {"SBFN": -2,   # Fernando de Noronha (figura en PE)
                  "SBTT": -5,   # Tabatinga (oeste de AM) - verificar
                  "SWEI": -5}   # Eirunepé (oeste de AM) - verificar
mapa_huso_uf = F.create_map(*[F.lit(x) for kv in HUSO_UF.items() for x in kv])
mapa_huso_exc = F.create_map(*[F.lit(x) for kv in HUSO_EXCEPCION.items() for x in kv])

a = aerodromos.select(F.col("codigo_oaci").alias("icao_aeropuerto"), F.col("nome").alias("nombre_anac"),
                      F.col("municipio").alias("municipio_anac"), "uf_sigla")
dim_aeropuerto = (codigos
    .join(claves_oa, codigos.icao_aeropuerto == claves_oa.clave, "left")
    .join(a, "icao_aeropuerto", "left")
    .withColumn("iso_pais", F.when(F.col("iso_country").isNotNull(), F.col("iso_country"))
                             .when(F.col("uf_sigla").isNotNull(), F.lit("BR")))
    .withColumn("uf", F.coalesce(F.when(F.col("iso_country") == "BR", F.substring("iso_region", 4, 2)), F.col("uf_sigla")))
    .withColumn("huso_utc", F.when(F.col("iso_pais") == "BR",
                                   F.coalesce(mapa_huso_exc[F.col("icao_aeropuerto")], mapa_huso_uf[F.col("uf")], F.lit(-3))))
    .withColumn("tamano", F.coalesce(mapa_tamano[F.col("type")],
                                     F.when(F.col("type").isNotNull(), F.lit("OTRO")), F.lit("DESCONOCIDO")))
    .select("icao_aeropuerto", F.coalesce("name", "nombre_anac").alias("nombre_aeropuerto"),
            F.col("type").alias("tipo_ourairports"), "tamano",
            F.coalesce("municipality", "municipio_anac").alias("ciudad"), "uf", "iso_pais",
            "huso_utc", "latitud", "longitud", "iata_code"))
print("Aeropuertos:", dim_aeropuerto.count())
dim_aeropuerto.groupBy("tamano").count().orderBy(F.desc("count")).show()
dim_aeropuerto.filter("iso_pais IS NULL OR tamano = 'DESCONOCIDO'").show(truncate=False)
dim_aeropuerto.filter("iso_pais = 'BR'").groupBy("huso_utc").count().orderBy("huso_utc").show()

In [ ]:
# dim_pais y dim_fecha
CONTINENTE = {"AF": "África", "AN": "Antártida", "AS": "Asia", "EU": "Europa",
              "NA": "América del Norte", "OC": "Oceanía", "SA": "América del Sur"}
mapa_cont = F.create_map(*[F.lit(x) for kv in CONTINENTE.items() for x in kv])
dim_pais = countries.select("iso_pais", "nombre_pais", mapa_cont[F.col("continent")].alias("continente"))

DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
mapa_dias = F.create_map(*[F.lit(x) for i, d in enumerate(DIAS, 1) for x in (i, d)])
dia_iso = lambda c: ((F.dayofweek(c) + 5) % 7) + 1          # 1 = lunes ... 7 = domingo
dim_fecha = (vuelos.select(F.to_date("fecha_referencia").alias("fecha")).filter("fecha IS NOT NULL").distinct()
             .select("fecha", F.year("fecha").alias("anio"), F.month("fecha").alias("mes"),
                     F.date_format("fecha", "yyyy-MM").alias("anio_mes"),
                     F.quarter("fecha").alias("trimestre"),
                     dia_iso("fecha").alias("dia_semana_num"),
                     mapa_dias[dia_iso("fecha")].alias("dia_semana"),
                     F.weekofyear("fecha").alias("semana_anio"),
                     (dia_iso("fecha") >= 6).alias("es_fin_de_semana")))
print("Países:", dim_pais.count(), "| Días:", dim_fecha.count())

In [ ]:
# fact_vuelo: solo pasajeros (N e I), con hora local de salida
huso_origen = dim_aeropuerto.select(F.col("icao_aeropuerto").alias("icao_origem"), F.col("huso_utc").alias("huso_origen"))
fact_vuelo = (vuelos.filter(F.col("tipo_linha").isin("N", "I"))
    .join(huso_origen, "icao_origem", "left")
    .withColumn("partida_prevista_local",
                F.when(F.col("huso_origen").isNotNull(),
                       F.col("partida_prevista") + F.expr("make_interval(0,0,0,0, huso_origen + 3, 0, 0)")))
    .withColumn("id_vuelo", F.sha2(F.concat_ws("|", *[F.col(c).cast("string") for c in clave_vuelo]), 256))
    .withColumn("puntual", F.when(F.col("realizado") & F.col("atraso_partida_min").isNotNull(),
                                  F.col("atraso_partida_min") <= 15))
    .withColumn("atrasado_15", F.when(F.col("puntual").isNotNull(), ~F.col("puntual")))
    .select("id_vuelo", "icao_empresa", "numero_voo", "codigo_di", "tipo_linha",
            (F.col("tipo_linha") == "I").alias("es_internacional"),
            "icao_origem", "icao_destino",
            F.to_date("fecha_referencia").alias("fecha"), "tiene_horario_previsto",
            "partida_prevista", "partida_real", "chegada_prevista", "chegada_real",
            F.hour("partida_prevista").alias("hora_prevista_brasilia"),
            "partida_prevista_local",
            F.hour("partida_prevista_local").alias("hora_local"),
            dia_iso("partida_prevista_local").alias("dia_semana_local"),
            "situacao_voo", "realizado", "cancelado", "informado",
            F.round("atraso_partida_min", 1).alias("atraso_partida_min"),
            F.round("atraso_chegada_min", 1).alias("atraso_chegada_min"),
            "puntual", "atrasado_15", "atraso_extremo",
            "anio_mes"))                      # la columna de partición va última
print("Vuelos en fact_vuelo:", fact_vuelo.count())
fact_vuelo.groupBy("anio_mes", "tipo_linha").count().orderBy("anio_mes", "tipo_linha").show(30)

Comprobación para P3: si los 10 aeropuertos con más movimiento están todos en UTC−3, la hora de Brasília ya es la hora local y la decisión del grupo queda justificada con los datos.

In [ ]:
movimientos = (fact_vuelo.select(F.col("icao_origem").alias("icao_aeropuerto"))
               .union(fact_vuelo.select(F.col("icao_destino").alias("icao_aeropuerto")))
               .groupBy("icao_aeropuerto").count())
(movimientos.join(dim_aeropuerto, "icao_aeropuerto")
 .select("icao_aeropuerto", "nombre_aeropuerto", "uf", "huso_utc", "tamano", "count")
 .orderBy(F.desc("count")).show(10, truncate=40))

In [ ]:
# Guardar el modelo en su ubicación dentro de la zona refinada
modelo = {"dim_empresa": dim_empresa, "dim_aeropuerto": dim_aeropuerto, "dim_pais": dim_pais,
          "dim_fecha": dim_fecha}
for nombre, df in modelo.items():
    df.write.mode("overwrite").parquet(f"{MODELO}/{nombre}")
fact_vuelo.write.mode("overwrite").partitionBy("anio_mes").parquet(f"{MODELO}/fact_vuelo")
modelo["fact_vuelo"] = fact_vuelo
print("Modelo guardado en", MODELO)

## 7. Tablas de Hive

Todas son **EXTERNAL** con `LOCATION` explícito apuntando al parquet del modelo: si se hace `DROP TABLE`, los datos quedan en HDFS. No se usa CTAS, porque crea tablas internas en `/user/hive/warehouse`.

El DDL se arma a partir del esquema de cada DataFrame, así los tipos de Hive coinciden siempre con los del parquet. `fact_vuelo` se particiona por `anio_mes`; como Spark ya escribió las carpetas `anio_mes=...`, `MSCK REPAIR TABLE` registra las particiones.

Si `CREATE DATABASE` da *Permission denied* sobre `/user`, correr en la VM:
`docker exec namenode hdfs dfs -mkdir -p /user/hive/warehouse` y `docker exec namenode hdfs dfs -chmod -R 777 /user/hive`.

In [ ]:
def tipo_hive(dt):
    return {"StringType": "STRING", "IntegerType": "INT", "LongType": "BIGINT", "DoubleType": "DOUBLE",
            "BooleanType": "BOOLEAN", "TimestampType": "TIMESTAMP", "DateType": "DATE"}[type(dt).__name__]

def ddl_externa(nombre, df, ubicacion, particion=None):
    cols = ",\n  ".join(f"`{f.name}` {tipo_hive(f.dataType)}" for f in df.schema.fields if f.name != particion)
    part = f"\nPARTITIONED BY (`{particion}` STRING)" if particion else ""
    return (f"CREATE EXTERNAL TABLE {DB}.{nombre} (\n  {cols}\n){part}\n"
            f"STORED AS PARQUET\nLOCATION '{ubicacion}'")

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB}")
ddls = []
for nombre, df in modelo.items():
    particion = "anio_mes" if nombre == "fact_vuelo" else None
    ddl = ddl_externa(nombre, df, f"{MODELO}/{nombre}", particion)
    spark.sql(f"DROP TABLE IF EXISTS {DB}.{nombre}")   # externa: no borra los datos
    spark.sql(ddl)
    ddls.append(ddl + ";")
spark.sql(f"MSCK REPAIR TABLE {DB}.fact_vuelo")

print("\n\n".join(ddls))   # este texto va al repo como sql/ddl_modelo.sql

## 8. Validación final en Hive

Para cada tabla: filas en Hive contra filas del DataFrame, tipo de tabla y ubicación. Estas salidas (más una consulta en Hue) son la evidencia para el informe.

In [ ]:
spark.sql(f"SHOW TABLES IN {DB}").show()
for nombre, df in modelo.items():
    en_hive, en_df = spark.table(f"{DB}.{nombre}").count(), df.count()
    print(f"[{'OK' if en_hive == en_df else 'DIFERENTE'}] {DB}.{nombre}: Hive {en_hive:,} / DataFrame {en_df:,}")
spark.sql(f"SHOW PARTITIONS {DB}.fact_vuelo").show(24, truncate=False)
(spark.sql(f"DESCRIBE FORMATTED {DB}.fact_vuelo")
 .filter(F.col("col_name").isin("Type", "Location", "Serde Library", "InputFormat"))
 .show(truncate=False))

In [ ]:
# Consulta de prueba sobre el modelo, como la va a usar el notebook 2
spark.sql(f'''
SELECT e.nacionalidad,
       COUNT(*)                                        AS vuelos,
       ROUND(AVG(CASE WHEN f.cancelado THEN 1 ELSE 0 END) * 100, 2) AS pct_cancelados,
       ROUND(AVG(CASE WHEN f.puntual THEN 1.0 WHEN f.puntual = false THEN 0.0 END) * 100, 2) AS pct_puntuales
FROM {DB}.fact_vuelo f
JOIN {DB}.dim_empresa e ON f.icao_empresa = e.icao_empresa
GROUP BY e.nacionalidad
ORDER BY vuelos DESC
''').show()